# scRareBench | High-Level MrVI | Dataset 0 — BMMC (GSE194122 curated)

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This Google Colab notebook runs **MrVI** on scRareBench Dataset 0 (`gse194122`) through the generic **MethodSpec → benchmark_method** API.

The method stays user-side. scRareBench loads the curated benchmark, runs independent method seeds, validates latent/cell alignment, computes metrics, and creates the multi-seed delivery.

### Dataset 0 / MrVI contract
- dataset selector: `0` (`gse194122`)
- curated cells: 89,199
- benchmark label: `celltype`
- benchmark/evaluation batch: `BATCH`
- MrVI `sample_key`: `BATCH`
- nuisance `batch_key`: none
- evaluated representation: MrVI **u latent** (`give_z=False`)
- method-side HVGs: 4,000 batch-aware Seurat-v3 HVGs from raw counts, `BATCH`, `span=0.3`
- method seeds: `42, 123, 2026`
- benchmark seed: fixed at `42`

`celltype` is not supplied to MrVI and is used only during benchmarking.

> MrVI is expensive on 89k cells. Use a **GPU + High-RAM** Colab runtime. For a smoke test, temporarily change `METHOD_SEEDS` to `[42]`.

## 1. Install scRareBench and MrVI dependencies


In [ ]:
import subprocess, sys

SCRAREBENCH_GITHUB = "git+https://github.com/amirhossein-alishahi/scRareBench_.git@cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--no-deps", SCRAREBENCH_GITHUB
])

METHOD_DEPENDENCIES = ("scvi-tools==1.4.3",)
INSTALL_METHOD_DEPENDENCIES = True

from scrarebench.runtime import print_install_report, setup_runtime
install_report = setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES if INSTALL_METHOD_DEPENDENCIES else (),
    extra_imports=("scvi", "scvi.external") if INSTALL_METHOD_DEPENDENCIES else (),
    quiet=False,
)
print_install_report(install_report)


In [ ]:
# OPTIONAL: informational runtime diagnostics only.
# Exact matching is NOT required.
#
# import sys
# from importlib import metadata
# print(sys.version)
# _EXPECTED_COLAB_ANCHORS = {"numpy":"2.0.2", "torch":"2.11.0", "jax":"0.7.2"}
# for pkg, expected in _EXPECTED_COLAB_ANCHORS.items():
#     try: observed = metadata.version(pkg)
#     except metadata.PackageNotFoundError: observed = "not installed"
#     print(pkg, "validated:", expected, "current:", observed)
# print("scvi-tools:", metadata.version("scvi-tools"))


## 2. Load curated Dataset 0


In [ ]:
from scrarebench import load_dataset, dataset_info

adata = load_dataset(0)
info = dataset_info(adata)

print(adata)
print(info)

assert adata.n_obs == 89_199, adata.n_obs
assert info["label_key"] == "celltype"
assert info["batch_key"] == "BATCH"


## 3. Seeds and validated MrVI configuration

Each seed trains a separate model from scratch. The benchmark seed is held fixed so cross-seed variation reflects method stochasticity rather than evaluation randomness.


In [ ]:
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a quick smoke test
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    "n_hvg": 4000,
    "hvg_span": 0.3,
    "sample_key": "BATCH",
    "nuisance_batch_key": None,
    "n_latent_u": 30,
    "n_latent_z": 30,
    "encoder_n_hidden": 128,
    "encoder_n_layers": 2,
    "max_epochs": 300,
    "train_size": 0.90,
    "batch_size": 128,
    "early_stopping": True,
    "early_stopping_patience": 25,
}

OUTPUT_DIR = "/content/scrarebench_mrvi_dataset0_highlevel"


## 4. User-owned MrVI runner

Pipeline per seed:

`raw counts → batch-aware Seurat-v3 HVGs → MrVI(sample_key=BATCH) → posterior-mean u latent`

The benchmark label is not used by the method.


In [ ]:
import gc
import numpy as np
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
import torch
import scvi
from scvi.external import MRVI
from scrarebench import MethodOutput


def _looks_like_raw_counts(matrix, max_values=100_000, seed=42):
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix).ravel()
    values = np.asarray(values)
    if len(values) > max_values:
        rng = np.random.default_rng(seed)
        values = values[rng.choice(len(values), size=max_values, replace=False)]
    values = values.astype(float, copy=False)
    if len(values) == 0:
        return True
    if not np.isfinite(values).all():
        return False
    return values.min(initial=0.0) >= 0 and np.allclose(values, np.rint(values), atol=1e-6)


def run_mrvi(method_adata, seed, config):
    batch_key = info["batch_key"]
    count_layer = info.get("count_layer") or "counts"

    if "feature_types" in method_adata.var.columns:
        feature_types = method_adata.var["feature_types"].astype(str).str.upper()
        rna_mask = feature_types.eq("GEX").to_numpy()
        if rna_mask.sum() == 0:
            raise ValueError("No GEX features found.")
        if rna_mask.sum() != method_adata.n_vars:
            method_adata = method_adata[:, rna_mask].copy()

    method_adata.var_names_make_unique()

    if count_layer not in method_adata.layers:
        raise KeyError(
            f"Raw-count layer {count_layer!r} is required. "
            f"Available layers: {list(method_adata.layers)}"
        )
    if not _looks_like_raw_counts(method_adata.layers[count_layer], seed=int(seed)):
        raise ValueError(f"Layer {count_layer!r} does not look like raw counts.")

    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), method_adata.n_vars),
        batch_key=batch_key,
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )

    hvg_mask = method_adata.var["highly_variable"].fillna(False).to_numpy()
    if hvg_mask.sum() == 0:
        raise RuntimeError("No HVGs selected.")

    counts_hvg = method_adata.layers[count_layer][:, hvg_mask]
    if sp.issparse(counts_hvg):
        counts_hvg = counts_hvg.tocsr().astype(np.float32)
    else:
        counts_hvg = np.asarray(counts_hvg, dtype=np.float32)

    adata_mrvi = ad.AnnData(
        X=counts_hvg.copy(),
        obs=method_adata.obs.copy(),
        var=method_adata.var.loc[hvg_mask].copy(),
    )
    adata_mrvi.var_names_make_unique()

    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    scvi.settings.seed = int(seed)

    setup_kwargs = {
        "sample_key": str(config["sample_key"]),
        "backend": "torch",
    }
    if config.get("nuisance_batch_key") is not None:
        setup_kwargs["batch_key"] = str(config["nuisance_batch_key"])

    MRVI.setup_anndata(adata_mrvi, **setup_kwargs)

    model = MRVI(
        adata_mrvi,
        n_latent=int(config["n_latent_z"]),
        n_latent_u=int(config["n_latent_u"]),
        encoder_n_hidden=int(config["encoder_n_hidden"]),
        encoder_n_layers=int(config["encoder_n_layers"]),
    )

    model.train(
        max_epochs=int(config["max_epochs"]),
        accelerator="auto",
        devices="auto",
        train_size=float(config["train_size"]),
        validation_size=1.0 - float(config["train_size"]),
        batch_size=int(config["batch_size"]),
        early_stopping=bool(config["early_stopping"]),
        early_stopping_patience=int(config["early_stopping_patience"]),
        check_val_every_n_epoch=1,
        enable_progress_bar=True,
    )

    latent = np.asarray(
        model.get_latent_representation(
            adata=adata_mrvi,
            batch_size=int(config["batch_size"]),
            use_mean=True,
            give_z=False,
        ),
        dtype=np.float32,
    )

    expected = (adata_mrvi.n_obs, int(config["n_latent_u"]))
    if latent.shape != expected:
        raise RuntimeError(f"Unexpected MrVI u latent shape: {latent.shape}; expected {expected}.")
    if not np.isfinite(latent).all():
        raise ValueError("MrVI latent contains NaN or infinite values.")

    barcodes = adata_mrvi.obs_names.copy()

    del model, adata_mrvi, counts_hvg
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return MethodOutput(
        latent=latent,
        barcodes=barcodes,
        representation_key="X_mrvi_u",
    )


## 5. Run the High-Level multi-seed benchmark


In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="MrVI-u",
    runner=run_mrvi,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir=OUTPUT_DIR,
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "run_scib": True,
        "scib_n_hvg": 4000,
        "scib_reference_n_pcs": 50,
        "scib_n_jobs": 1,
        "scib_progress_bar": True,
        "scib_include_silhouette_batch": True,
        "scib_require_success": True,
        "include_latent_in_bundle": False,
    },
    install_dependencies=False,
    finalize=True,
)


## 6. Inspect summary and output files


In [ ]:
from IPython.display import display

display(result.summary().round(5))
print("Seeds:", result.seeds)
print("Multi-seed report:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(f"Seed {seed}: {run.output_dir}")


## 7. Download the final multi-seed archive


In [ ]:
from pathlib import Path
from google.colab import files

archive_path = Path(result.archive_path)
if not archive_path.exists():
    raise FileNotFoundError(
        f"Output archive was not found: {archive_path}\n"
        "Make sure benchmark_method(..., finalize=True) completed successfully."
    )

print(f"Downloading: {archive_path.name}")
print(f"Size: {archive_path.stat().st_size / (1024**2):.2f} MB")
files.download(str(archive_path))
